In [1]:
%%time

import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool, EShapCalcType, EFeaturesSelectionAlgorithm
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, f1_score
from sklearn.pipeline import make_pipeline,make_union
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures, PowerTransformer, FunctionTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, KFold, ShuffleSplit, learning_curve, GridSearchCV
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
pd.set_option("display.max_column", 100)

Wall time: 1min


In [173]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
ss = pd.read_csv('solution_format.csv')

train = train.drop("ID", axis=1)

In [179]:
df = train.copy()
grp_1 = train.filter(regex="_EventInd|index_").columns.to_list()
grp_2 = train.filter(regex="_A|_F|_G").columns.to_list()
grp_3 = list(set(train.columns).difference(grp_1).difference(grp_2))
df[grp_2] = df[grp_2].astype("object")

In [243]:
from sklearn.preprocessing import FunctionTransformer

def identity_func(X):
    X[grp_1] = X[grp_1].apply(np.float64)
    X[[x for x in grp_3 if x!="Target"]] = X[[x for x in grp_3 if x!="Target"]].apply(np.float64)
    return X

In [235]:
process = 0

def imputation(df, i):

    global process

    if i==1:
        grp_1_pipeline=make_pipeline(SimpleImputer(strategy="constant", fill_value=0))
        #grp_2_pipeline=make_pipeline(SimpleImputer(missing_values='NAN', strategy="most_frequent"))
        grp_2_pipeline=make_pipeline(SimpleImputer(strategy="most_frequent"))
        grp_3_pipeline=make_pipeline(SimpleImputer(strategy="mean"))
        process=make_column_transformer((grp_1_pipeline, grp_1), (grp_2_pipeline, grp_2), (grp_3_pipeline, [x for x in grp_3 if x!="Target"]))
        # numerical_pipeline=make_union(SimpleImputer(strategy="constant", fill_value=1234), MissingIndicator())
        # categorical_pipeline=make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown='ignore'))
        process.fit(df)
        data=process.transform(df)
    else:
        data=process.transform(df)
    
    return data

In [236]:
# # Définir une fonction qui renvoie simplement les données sans les modifier


# # Créer un transformateur d'identité
# identity_transformer = 

# # Appliquer le transformateur d'identité à vos données
# X_transformed = identity_transformer.fit_transform(X)

In [237]:
def pre_processing(df):

    #df=create_new_features(df)
    #df=encodage(df)

    if "Target" in df.columns.values:
        X = df.drop("Target",axis=1)
        y = df.Target
        return X,y
    else:
        X = df
        return X

In [238]:
X,y=pre_processing(df)

In [239]:
cv=KFold(n_splits=5,shuffle=True,random_state=0)

In [240]:
RANDOM_SEED=1234

In [241]:
pd.DataFrame(imputation(X,i=1), columns=[*grp_1, *grp_2, *[x for x in grp_3 if x!="Target"]])

,TR_1_EventInd,TR_2_EventInd,TR_3_EventInd,index_1,index_2,index_3,feature_10_A,feature_10_F,feature_10_G,feature_1_A,feature_1_F,feature_1_G,feature_2_A,feature_2_F,feature_2_G,feature_3_A,feature_3_F,feature_3_G,feature_4_A,feature_4_F,feature_4_G,feature_5_A,feature_5_F,feature_5_G,feature_6_A,feature_6_F,feature_6_G,feature_7_A,feature_7_F,feature_7_G,feature_8_A,feature_8_F,feature_8_G,feature_9_A,feature_9_F,feature_9_G,feature_X_A,feature_5_C,feature_3_D,feature_1_E,feature_6_E,feature_X_D,feature_2_E,feature_9_B,feature_7_D,feature_5_E,feature_4_B,feature_7_B,feature_3_E,feature_9_E,feature_4_E,feature_10_B,feature_6_C,feature_1_D,feature_2_D,feature_X_C,feature_7_E,feature_8_B,feature_3_C,feature_9_D,feature_1_C,feature_3_B,feature_8_C,feature_5_B,feature_5_D,feature_8_D,feature_6_B,feature_2_B,feature_7_C,feature_1_B,feature_9_C,feature_6_D,feature_4_C,feature_8_E,feature_X_B,feature_2_C
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1,0.344858,-0.00643,143.18432,221.510229,-0.274727,51.036046,0.061334,-0.033399,110.354351,0.006647,0.011449,160.359302,3.975986,117.22327,0.02789,0.326615,-0.024525,0.000024,1.542456,314.063895,0.040916,0.144423,-0.004353,0.093747,0.016414,0.119476,0.033782,-0.012599,-0.006211,0.014884,0.010248,0.319489,0.008254,-0.032864,-0.011029,0.108756,104.270913,0.049,0.081602
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1,0.344858,-0.00643,143.18432,221.510229,-1.140626,51.036046,0.061334,-0.033399,110.354351,0.006647,0.011449,160.359302,3.975986,117.22327,0.02789,0.326615,-0.024525,0.000024,1.287473,314.063895,0.040916,0.144423,-0.004353,0.093747,0.016414,0.119476,0.033782,-0.012599,-0.006211,0.014884,0.010248,0.319489,0.008254,-0.032864,-0.011029,0.108756,104.270913,0.015,0.081602
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1,-1.070837,0.381673,122.793643,110.291715,-1.221567,41.993286,0.032,0.528526,11.492357,0.01,0.01,114.972429,2.161857,106.70185,0.023,-0.759255,-1.354808,1.293117,0.251501,209.787785,0.016,-1.728743,0.729995,-0.224414,0.01,-0.886497,0.018,-0.602141,-0.940992,0.009,0.011,-1.770791,0.004,-1.938003,0.242138,0.120834,14.801224,0.006,-1.290421
3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,2.0,0.0,1.0,5.0,1.0,1.0,1.0,1.0,1.0,2.0,0.0,0.0,0.0,1,3.357983,-0.935754,116.5155,125.405357,0.527796,45.550786,0.033,-0.760985,24.703054,0.004,0.003,134.0235,2.4165,112.413493,0.019,1.188867,-1.497321,0.104815,1.673783,228.473857,0.023,0.077801,-0.982337,-1.372548,0.012,1.647273,0.068,3.27753,-0.377874,0.004,0.005,1.044063,0.005,0.657038,-0.07934,1.579835,16.205176,0.039,1.891042
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,0.0,2.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,2.0,0,0.732436,0.112219,123.708286,156.600929,-0.301984,56.961143,0.084,-0.618166,52.272714,0.004,0.011,150.783072,3.438571,117.22327,0.023,-1.111772,2.354236,0.776717,0.263406,263.827785,0.039,0.25702,1.275223,1.664108,0.013,-1.113186,0.019,-0.897212,0.148994,0.015,0.014,-1.081073,0.014,2.281595,0.260519,0.7826,22.298981,0.037,0.811079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2655,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,0.0,1.0,2.0,1.0,0.0,1.0,1.0,1.0,2.0,0.0,1.0,4.0,0.0,1.0,2.0,1.0,1.0,1.0,1.0,0.0,1.0,1,1.911614,0.789927,133.888928,187.967928,-0.009556,50.097857,0.059,0.900047,42.409232,0.004,0.003,152.946857,4.668214,117.22327,0.026,1.830395,-0.819746,

In [244]:
identity_func(pd.DataFrame(imputation(X,i=1), columns=[*grp_1, *grp_2, *[x for x in grp_3 if x!="Target"]])).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2660 entries, 0 to 2659
Data columns (total 76 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TR_1_EventInd  2660 non-null   float64
 1   TR_2_EventInd  2660 non-null   float64
 2   TR_3_EventInd  2660 non-null   float64
 3   index_1        2660 non-null   float64
 4   index_2        2660 non-null   float64
 5   index_3        2660 non-null   float64
 6   feature_10_A   2660 non-null   object 
 7   feature_10_F   2660 non-null   object 
 8   feature_10_G   2660 non-null   object 
 9   feature_1_A    2660 non-null   object 
 10  feature_1_F    2660 non-null   object 
 11  feature_1_G    2660 non-null   object 
 12  feature_2_A    2660 non-null   object 
 13  feature_2_F    2660 non-null   object 
 14  feature_2_G    2660 non-null   object 
 15  feature_3_A    2660 non-null   object 
 16  feature_3_F    2660 non-null   object 
 17  feature_3_G    2660 non-null   object 
 18  feature_

In [245]:
model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=8, loss_function='Logloss', custom_metric=['AUC'], random_seed=RANDOM_SEED)                

scores=[]
for i,j in enumerate(cv.split(X,y)):
    tr = identity_func(pd.DataFrame(imputation(X.iloc[j[0]],i=1), columns=[*grp_1, *grp_2, *[x for x in grp_3 if x!="Target"]]))
    #tr[grp_2] = tr[grp_2].apply(str)

    ev = identity_func(pd.DataFrame(imputation(X.iloc[j[1]],i=1252), columns=[*grp_1, *grp_2, *[x for x in grp_3 if x!="Target"]]))
    #ev[grp_2] = ev[grp_2].apply(str)

    train=Pool(
        data=tr,
        label=y.iloc[j[0]],
        cat_features=grp_2
    )
    eval=Pool(
        data=ev,
        label=y.iloc[j[1]],
        cat_features=grp_2
    )
    model.fit(
        train,
        eval_set=eval,
        verbose=False
    )
    
    y_pred=model.predict(X.iloc[j[1]])
    score=f1_score(y.iloc[j[1]], y_pred, average='binary')
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

CatBoostError: Invalid type for cat_feature[non-default value idx=0,feature_idx=6]=0.0 : cat_features must be integer or string, real number values and NaN values should be converted to string.

In [8]:

NUM_THREADS=0
MODEL_PARAMS_LGB = {
    "boosting_type": 'gbdt',
    "objective": "regression",  # <= set objective
    "metric":None,
    "seed": RANDOM_SEED,
    "verbose": -1,
}   
FIT_PARAMS_LGB = {"early_stopping_rounds": 100, "verbose_eval":100}



In [141]:
df.feature_10_G.isna().sum()

448

In [148]:
imputer = SimpleImputer(strategy="most_frequent")
np.isnan(imputer.fit_transform(df.feature_10_G.to_numpy().reshape(-1,1))).sum()

0

In [ ]:
params = {
        "subsample": trial.suggest_float("subsample", 0.5, 0.99),
        'od_wait': trial.suggest_int('od_wait', 10, 50, step=2),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 0.99),
        "random_strength": trial.suggest_int("random_strength", 1, 10, step=1),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.5),
        "max_depth": trial.suggest_int("max_depth", 4, 15, step=1),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000, step=1),
        'learning_rate': trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        }